# Big Data Analytics — Assignment 02
> Author : Badr TAJINI - Big Data Analytics - ESIEE 2025-2026


**Chapter 3 :** From MapReduce → Spark patterns  
**Chapter 4 :** Text analysis in PySpark

**Tools :** Spark or PySpark.   
**Advice:** Keep evidence and reproducibility.

## 0. Bootstrap

In [1]:
# write some code here
# - create SparkSession('BDA-A02') with UTC timezone
# - print Spark/PySpark/Python versions
# - set spark.sql.shuffle.partitions to a small value for local runs

import sys
import platform
from pyspark.sql import SparkSession
import pyspark

spark = (
    SparkSession.builder
    .appName("BDA-Assignment02")  
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16") 
    .getOrCreate()
)
sc = spark.sparkContext

# Affichage des versions
print(f"Spark Session créée | Port UI: {spark.sparkContext.uiWebUrl}")
print(f"Spark version: {spark.version}")
print(f"PySpark version: {pyspark.__version__}")
print(f"Python version: {sys.version.split()[0]} | OS: {platform.platform()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/17 11:24:34 WARN Utils: Your hostname, Elliot, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/17 11:24:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 11:24:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session créée | Port UI: http://10.255.255.254:4040
Spark version: 4.0.1
PySpark version: 4.0.1
Python version: 3.10.19 | OS: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39


In [2]:
print(spark.sparkContext.uiWebUrl)

http://10.255.255.254:4040


## 1. Dataset acquisition

In [3]:
# write some code here
# - ensure data/shakespeare.txt exists; if missing, download from the URL in the overview
# - create (a) an RDD of lines and (b) a DataFrame with column 'line'
# - show a few 
## 1. Load data (Local File)
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
PROOF_DIR = BASE_DIR / "proof"

for directory in (DATA_DIR, OUTPUTS_DIR, PROOF_DIR):
    directory.mkdir(exist_ok=True)


FILE_NAME = "shakespeare.txt" 
TEXT_PATH = DATA_DIR / FILE_NAME

if not TEXT_PATH.exists():
    raise FileNotFoundError(f"Erreur: Le fichier {TEXT_PATH} est introuvable. Veuillez le placer manuellement dans le dossier data/.")
    
raw_rdd = spark.sparkContext.textFile(str(TEXT_PATH)).cache()

lines_df = spark.read.text(str(TEXT_PATH)).withColumnRenamed("value", "line").cache()

raw_rdd.count()
lines_df.count()

print(f"Data loaded successfully from: {TEXT_PATH}")
lines_df.show(5, truncate=False)


Data loaded successfully from: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/data/shakespeare.txt
+----------------------+
|line                  |
+----------------------+
|1609                  |
|                      |
|THE SONNETS           |
|                      |
|by William Shakespeare|
+----------------------+
only showing top 5 rows


## 2. Tokenization helper

In [4]:
# write some code here
# - implement a tokenizer: lowercase, split on non-letters, drop empties
# - implement truncate(tokens, n=40) for PMI
import re
from pyspark import StorageLevel

TOKEN_PATTERN = re.compile(r"[a-z]+")

def tokenize(text: str):
    return TOKEN_PATTERN.findall(text.lower())

def truncate_tokens(tokens, n=40):
    return tokens[:n]

print("Tokenisation et mise en cache du RDD...")
tokenized_lines_rdd = (
    raw_rdd
    .map(tokenize)
    .filter(lambda tokens: len(tokens) > 0) 
    #.persist(StorageLevel.MEMORY_ONLY) 
)

#line_count = tokenized_lines_rdd.count()
#print(f"Tokenizé et mis en cache {line_count} lignes non-vides.")

#print("\n--- Exemple de 5 lignes tokenizées ---")
#for line_tokens in tokenized_lines_rdd.take(5):
#    print(line_tokens)

Tokenisation et mise en cache du RDD...


## 3. Part A — Bigram relative frequency (pairs)

In [5]:
# write some code here
# - emit ((w_i, w_{i+1}), 1) and ((w_i, '*'), 1)
# - reduceByKey to counts; compute relative frequency
# - write outputs/bigram_pairs_top.csv (top N)
# - save explain('formatted') from a DF stage to proof/plan_bigrams.txt

from operator import add
from io import StringIO
from contextlib import redirect_stdout
from pyspark.sql import functions as F


# 1. Compter les paires (w1, w2)
# Entrée: ['all', 'the', 'world', 's', 'a', 'stage']
# Sortie: ((all, the), 1), ((the, world), 1), ((world, s), 1), ...
pair_counts_rdd = (
    tokenized_lines_rdd
    .flatMap(lambda tokens: [((tokens[i], tokens[i + 1]), 1) for i in range(len(tokens) - 1)])
    .reduceByKey(add)
)

# 2. Compter les "marges" (w1, *)
# Entrée: ['all', 'the', 'world', 's', 'a', 'stage']
# Sortie: (all, 1), (the, 1), (world, 1), ...
# Note: On ne compte que les mots qui sont en position w1 (tous sauf le dernier)
marginal_counts_rdd = (
    tokenized_lines_rdd
    .flatMap(lambda tokens: [(tokens[i], 1) for i in range(len(tokens) - 1)])
    .reduceByKey(add)
)

# 3. Joindre et calculer la fréquence relative
# On prépare le RDD des paires pour la jointure
# ( (w1, w2), count ) -> ( w1, (w2, count) )
pairs_to_join_rdd = (
    pair_counts_rdd
    .map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
)

# On joint les paires (w1, (w2, count)) avec les marges (w1, total_count)
relative_freq_rdd = (
    pairs_to_join_rdd
    .join(marginal_counts_rdd)
    # kv = (w1, ( (w2, count), total_count) )
    .map(lambda kv: (kv[0], kv[1][0][0], kv[1][0][1] / kv[1][1], kv[1][0][1]))
    # Sortie: (w1, w2, rel_freq, count)
)

# 4. Conversion en DataFrame pour le tri et la sauvegarde
bigram_pairs_df = spark.createDataFrame(
    relative_freq_rdd, 
    schema=["w1", "w2", "rel_freq", "count"]
)

# 5. Trier et prendre le Top 50
pairs_top_df = (
    bigram_pairs_df
    .orderBy(F.desc("rel_freq"), F.desc("count"), F.asc("w1"), F.asc("w2"))
    .limit(50)
)

(pairs_top_df
    .toPandas()
    .to_csv(OUTPUTS_DIR / "bigram_pairs_top.csv", index=False)
)
print(f"Résultats sauvegardés dans {OUTPUTS_DIR / 'bigram_pairs_top.csv'}")

plan_buffer = StringIO()
with redirect_stdout(plan_buffer):
    pairs_top_df.explain("formatted")
(PROOF_DIR / "plan_bigrams_pairs.txt").write_text(plan_buffer.getvalue())
print(f"Plan sauvegardé dans {PROOF_DIR / 'plan_bigrams_pairs.txt'}")

--- Démarrage de la Partie A (Pairs) ---


Sauvegarde du CSV des 'pairs'...


[Stage 13:>                                                         (0 + 4) / 4]

Résultats sauvegardés dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/outputs/bigram_pairs_top.csv
Sauvegarde du plan d'exécution...
Plan sauvegardé dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/proof/plan_bigrams_pairs.txt
--- Fin de la Partie A (Pairs) ---


## 4. Part A — Bigram relative frequency (stripes)

In [5]:
# write some code here
# - build stripes: w_i -> dict{ w_{i+1}: count }, merge with reduceByKey
# - normalize inside each stripe; write outputs/bigram_stripes_top.csv
from collections import Counter
from io import StringIO
from contextlib import redirect_stdout
from pyspark.sql import functions as F

def build_stripes(tokens: list[str]):
    stripes = {}
    for i in range(len(tokens) - 1):
        w1 = tokens[i]
        w2 = tokens[i + 1]
        
        # Crée le dictionnaire pour w1 s'il n'existe pas
        if w1 not in stripes:
            stripes[w1] = Counter()
        
        # Incrémente le compteur pour w2
        stripes[w1][w2] += 1
        
    # Émet une liste de (clé, dictionnaire)
    return [(head, counter) for head, counter in stripes.items()]

def merge_counters(c1: Counter, c2: Counter) -> Counter:
    c1.update(c2)
    return c1

def normalize_stripe(kv_pair: tuple):
    w1, stripe_counts = kv_pair
    
    # Calcule le total des transitions sortant de w1
    total_count = sum(stripe_counts.values())
    
    # Normalise chaque entrée (w1, w2) par le total de w1
    for w2, count in stripe_counts.items():
        rel_freq = count / total_count
        yield (w1, w2, rel_freq, count)

# 1. Construire les "stripes" partielles (par ligne) et les fusionner
#    (w1, Counter({'w2_a': 1})) + (w1, Counter({'w2_b': 1})) -> (w1, Counter({'w2_a': 1, 'w2_b': 1}))
stripes_rdd = (
    tokenized_lines_rdd
    .flatMap(build_stripes)
    .reduceByKey(merge_counters)
)

# 2. Normaliser et "aplatir" les stripes en lignes
#    (w1, Counter({'w2_a': 2, 'w2_b': 3})) ->
#    [ (w1, w2_a, 0.4, 2),
#      (w1, w2_b, 0.6, 3) ]
relative_freq_stripes_rdd = stripes_rdd.flatMap(normalize_stripe)

# 3. Conversion en DataFrame
bigram_stripes_df = spark.createDataFrame(
    relative_freq_stripes_rdd,
    schema=["w1", "w2", "rel_freq", "count"]
)

# 4. Trier et prendre le Top 50
stripes_top_df = (
    bigram_stripes_df
    .orderBy(F.desc("rel_freq"), F.desc("count"), F.asc("w1"), F.asc("w2"))
    .limit(50)
)

# 5. Action: Sauvegarder le CSV (c'est ce job qu'on va mesurer)
(stripes_top_df
    .toPandas()
    .to_csv(OUTPUTS_DIR / "bigram_stripes_top.csv", index=False)
)
print(f"Résultats sauvegardés dans {OUTPUTS_DIR / 'bigram_stripes_top.csv'}")

# 6. Sauvegarder le plan d'exécution (similaire à la Partie A)
plan_buffer_stripes = StringIO()
with redirect_stdout(plan_buffer_stripes):
    stripes_top_df.explain("formatted")
(PROOF_DIR / "plan_bigrams_stripes.txt").write_text(plan_buffer_stripes.getvalue())
print(f"Plan sauvegardé dans {PROOF_DIR / 'plan_bigrams_stripes.txt'}")



[Stage 9:>                                                          (0 + 2) / 2]

Résultats sauvegardés dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/outputs/bigram_stripes_top.csv
Plan sauvegardé dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/proof/plan_bigrams_stripes.txt


## 5. Part B — PMI with threshold K

In [13]:
# write some code here
# - keep only first 40 tokens per line
# - compute counts for x and (x,y); PMI = log10( P(x,y) / (P(x)*P(y)) )
# - --threshold K to filter low-frequency pairs
# - write outputs/pmi_pairs_sample.csv (or stripes version)
# - save proof/plan_pmi.txt if any DF stages are used
import math
from operator import add
from io import StringIO
from contextlib import redirect_stdout
from pyspark.sql import functions as F

K_THRESHOLD = 10
N_TOKENS = 40

# 1. Tronquer les lignes à N_TOKENS
# Note: .persist() est utile car ce RDD sera utilisé 3 fois
truncated_rdd = (
    tokenized_lines_rdd
    .map(lambda tokens: truncate_tokens(tokens, n=N_TOKENS))
    .persist()
)

# 2. Compter les paires (w1, w2)
# RDD: [ ((w1, w2), count), ... ]
pair_counts_rdd = (
    truncated_rdd
    .flatMap(lambda tokens: [((tokens[i], tokens[i + 1]), 1) for i in range(len(tokens) - 1)])
    .reduceByKey(add)
)

# 3. Compter les marges (w1, *)
# RDD: [ (w1, count_w1_star), ... ]
marginal_w1_counts_rdd = (
    truncated_rdd
    .flatMap(lambda tokens: [(tokens[i], 1) for i in range(len(tokens) - 1)])
    .reduceByKey(add)
)

# 4. Compter les marges (*, w2)
# RDD: [ (w2, count_star_w2), ... ]
marginal_w2_counts_rdd = (
    truncated_rdd
    .flatMap(lambda tokens: [(tokens[i + 1], 1) for i in range(len(tokens) - 1)])
    .reduceByKey(add)
)

# 5. Calculer le nombre total de paires (N_pairs)
# C'est une action, N_pairs est maintenant un nombre sur le driver
N_pairs = marginal_w1_counts_rdd.map(lambda kv: kv[1]).sum()
print(f"Nombre total de paires (N_pairs) après troncature: {N_pairs}")

# 6. Collecter les comptes w2 et N_pairs pour les diffuser (broadcast)
# C'est une action, w2_counts_map est un dictionnaire Python sur le driver
print("Collecte des comptes marginaux de w2...")
w2_counts_map = marginal_w2_counts_rdd.collectAsMap()

# Diffusion des variables pour les rendre disponibles sur tous les workers
N_pairs_bcast = sc.broadcast(N_pairs)
w2_counts_bcast = sc.broadcast(w2_counts_map)
print("Variables N_pairs et w2_counts diffusées (broadcasted).")

# 7. Préparer le RDD des paires pour la jointure
# ( (w1, w2), count_w1_w2 ) -> ( w1, (w2, count_w1_w2) )
pairs_to_join_rdd = (
    pair_counts_rdd
    .map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
)

# 8. Joindre avec les comptes de w1
# ( w1, ( (w2, count_w1_w2), count_w1_star ) )
joined_rdd = pairs_to_join_rdd.join(marginal_w1_counts_rdd)

# 9. Définir la fonction de calcul PMI (qui utilise les broadcast)
def calculate_pmi(record):
    w1, ((w2, count_w1_w2), count_w1_star) = record
    
    # Appliquer le seuil K
    if count_w1_w2 < K_THRESHOLD:
        return None
    
    # Récupérer les variables diffusées
    N = N_pairs_bcast.value
    w2_counts = w2_counts_bcast.value
    
    count_star_w2 = w2_counts.get(w2)
    
    # Vérifier que nous avons toutes les données (évite les erreurs de division par zéro)
    if not count_star_w2 or count_w1_star == 0:
        return None
        
    # Calculer PMI
    # PMI = log10( P(x,y) / (P(x) * P(y)) )
    # PMI = log10( (Count(x,y) / N) / ( (Count(x,*) / N) * (Count(*,y) / N) ) )
    # PMI = log10( (Count(x,y) * N) / (Count(x,*) * Count(*,y)) )
    
    numerator = float(count_w1_w2 * N)
    denominator = float(count_w1_star * count_star_w2)
    
    if denominator == 0:
         return None
         
    pmi = math.log10(numerator / denominator)
    
    return (w1, w2, pmi, count_w1_w2)

# 10. Appliquer la fonction PMI et filtrer les 'None'
pmi_rdd = (
    joined_rdd
    .map(calculate_pmi)
    .filter(lambda x: x is not None)
)

# 11. Conversion en DataFrame
pmi_df = spark.createDataFrame(
    pmi_rdd,
    schema=["w1", "w2", "pmi", "count"]
)

# 12. Trier pour obtenir l'échantillon (les plus fortes associations) et limiter
pmi_top_df = (
    pmi_df
    .orderBy(F.desc("pmi"), F.desc("count"), F.asc("w1"), F.asc("w2"))
    .limit(100) # Prenons un échantillon de 100
)

# 13. Action: Sauvegarder le CSV
print(f"Sauvegarde de l'échantillon PMI (K={K_THRESHOLD}, N={N_TOKENS})...")
(pmi_top_df
    .toPandas()
    .to_csv(OUTPUTS_DIR / "pmi_pairs_sample.csv", index=False)
)
print(f"Résultats sauvegardés dans {OUTPUTS_DIR / 'pmi_pairs_sample.csv'}")

# 14. Sauvegarder le plan d'exécution
plan_buffer_pmi = StringIO()
with redirect_stdout(plan_buffer_pmi):
    pmi_top_df.explain("formatted")
(PROOF_DIR / "plan_pmi.txt").write_text(plan_buffer_pmi.getvalue())
print(f"Plan sauvegardé dans {PROOF_DIR / 'plan_pmi.txt'}")

# Nettoyer le RDD persisté
truncated_rdd.unpersist()

Nombre total de paires (N_pairs) après troncature: 797952
Collecte des comptes marginaux de w2...
Variables N_pairs et w2_counts diffusées (broadcasted).


Sauvegarde de l'échantillon PMI (K=10, N=40)...
Résultats sauvegardés dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/outputs/pmi_pairs_sample.csv
Plan sauvegardé dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/proof/plan_pmi.txt


PythonRDD[56] at RDD at PythonRDD.scala:56

## 6. Part C — Inverted index build

In [14]:
# write some code here
# - assign doc_id = line_number // 10
# - compute term frequencies ((term, doc_id), tf)
# - aggregate into postings per term; compute df
# - write Parquet to outputs/index_parquet/

from operator import add
from io import StringIO
from contextlib import redirect_stdout
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1. Calculer les fréquences des termes (TF) par document (doc_id)
#    Nous utilisons raw_rdd.zipWithIndex() pour obtenir le numéro de ligne
#    kv = (ligne_texte, numero_de_ligne)
doc_term_counts_rdd = (
    raw_rdd.zipWithIndex()
    .flatMap(lambda kv: [
        ((token, kv[1] // 10), 1)  # Clé = (term, doc_id)
        for token in tokenize(kv[0]) if token
    ])
    .reduceByKey(add)  # Somme pour obtenir TF
)
# Sortie: ((term, doc_id), tf)

# 2. Transformer pour grouper par terme
#    Mapper de ((term, doc_id), tf) -> (term, (doc_id, tf))
term_postings_rdd = (
    doc_term_counts_rdd
    .map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
)
# Sortie: (term, (doc_id, tf))

# 3. Fonction pour formater les listes de postings
def format_postings(postings_list):
    """
    Trie une liste de (doc_id, tf) par doc_id et la formate
    en une liste de dictionnaires.
    """
    # Trier par doc_id, qui est le premier élément du tuple (doc_id, tf)
    sorted_postings = sorted(list(postings_list), key=lambda item: item[0])
    
    # Convertir en liste de dictionnaires (pour correspondre au StructType)
    return [{"doc_id": doc_id, "tf": tf} for doc_id, tf in sorted_postings]

# 4. Grouper par terme et formater les postings
#    C'est ici que l'index inversé est réellement construit
index_rows_rdd = (
    term_postings_rdd
    .groupByKey()
    .mapValues(format_postings)
)
# Sortie: (term, [{"doc_id": id1, "tf": f1}, {"doc_id": id2, "tf": f2}, ...])

# 5. Définir le schéma pour le DataFrame
schema = T.StructType([
    T.StructField("term", T.StringType(), False),
    T.StructField("df", T.IntegerType(), False),  # Document Frequency
    T.StructField("postings", T.ArrayType(T.StructType([
        T.StructField("doc_id", T.IntegerType(), False),
        T.StructField("tf", T.IntegerType(), False),
    ])), False),
])

# 6. Créer le DataFrame final
#    Mapper (term, postings_list) -> (term, df, postings_list)
#    df (Document Frequency) est simplement la longueur de la liste des postings
index_df = spark.createDataFrame(
    index_rows_rdd.map(lambda kv: (kv[0], len(kv[1]), kv[1])),
    schema=schema
)

# 7. Action: Écrire le DataFrame en format Parquet
output_index_path = OUTPUTS_DIR / "index_parquet"
print(f"Écriture de l'index inversé vers {output_index_path}...")
(
    index_df.write
    .mode("overwrite")
    .parquet(str(output_index_path))
)
print("Écriture terminée.")

# 8. Afficher un échantillon et sauvegarder le plan d'exécution
print("Échantillon de l'index (trié par df décroissant):")
index_df.orderBy(F.desc("df"), F.asc("term")).show(10, truncate=50)

print("Sauvegarde du plan d'exécution de l'index...")
plan_buffer_index = StringIO()
with redirect_stdout(plan_buffer_index):
    # Nous utilisons une action (comme .count()) sur le DF pour générer le plan
    # mais .write.parquet() est l'action que nous mesurons.
    # Pour obtenir le plan du .write, nous devons l'analyser dans l'UI.
    # .explain() sur le DF avant l'écriture est une bonne approximation.
    index_df.explain("formatted")
(PROOF_DIR / "plan_index_build.txt").write_text(plan_buffer_index.getvalue())
print(f"Plan sauvegardé dans {PROOF_DIR / 'plan_index_build.txt'}")



Écriture de l'index inversé vers /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/outputs/index_parquet...


Écriture terminée.
Échantillon de l'index (trié par df décroissant):


+----+-----+--------------------------------------------------+
|term|   df|                                          postings|
+----+-----+--------------------------------------------------+
| and|10632|[{1, 1}, {2, 3}, {3, 3}, {4, 1}, {5, 2}, {6, 1}...|
| the|10301|[{0, 1}, {1, 3}, {2, 3}, {3, 1}, {4, 5}, {5, 1}...|
|  to| 9226|[{1, 3}, {2, 1}, {3, 2}, {5, 2}, {6, 2}, {7, 3}...|
|   i| 8982|[{17, 1}, {19, 2}, {20, 2}, {23, 6}, {24, 2}, {...|
|  of| 8615|[{2, 1}, {3, 2}, {4, 1}, {5, 4}, {6, 2}, {8, 1}...|
|   a| 7530|[{1, 1}, {2, 1}, {6, 1}, {8, 1}, {12, 1}, {14, ...|
|  in| 6918|[{2, 2}, {4, 1}, {5, 1}, {8, 1}, {9, 1}, {10, 1...|
|that| 6910|[{1, 2}, {4, 1}, {7, 1}, {8, 1}, {9, 2}, {10, 1...|
|  my| 6560|[{3, 2}, {17, 1}, {22, 1}, {23, 2}, {25, 1}, {2...|
| you| 6013|[{21, 6}, {22, 2}, {25, 2}, {26, 4}, {27, 3}, {...|
+----+-----+--------------------------------------------------+
only showing top 10 rows
Sauvegarde du plan d'exécution de l'index...
Plan sauvegardé dans /mnt/c/Users/

## 7. Part C — Boolean retrieval (AND / OR)

In [15]:
# write some code here
# - implement evaluate_and(terms) and evaluate_or(terms) using postings
# - ranking: sum(tf) or df-normalized tf
# - run 3–5 queries; write outputs/queries_and_results.md
import os
from collections import defaultdict
from pyspark.sql import functions as F

# 1. Charger l'index Parquet en mémoire
# Nous n'avons pas besoin des 'schema' car Parquet les stocke
try:
    index_df = spark.read.parquet(str(OUTPUTS_DIR / "index_parquet"))
except Exception as e:
    print(f"Erreur: Impossible de lire l'index depuis {OUTPUTS_DIR / 'index_parquet'}")
    print("Veuillez d'abord exécuter la cellule de construction de l'index (Partie D).")
    raise e

# 2. Action: Collecter l'index sur le driver dans un dictionnaire
# C'est une action Spark. Elle ramène toutes les données sur le nœud principal.
print("Collecte de l'index en mémoire (driver)...")
index_local = {row.term: row.postings for row in index_df.collect()}
print(f"Index chargé avec {len(index_local)} termes.")


# 3. Fonctions d'aide pour la recherche

def postings_to_dict(term: str) -> dict:
    """
    Récupère la liste de postings pour un terme depuis l'index local
    et la convertit en un dictionnaire {doc_id: tf} pour une recherche rapide.
    """
    # row.postings est une liste de Row(doc_id, tf)
    # Nous la transformons en {doc_id: tf}
    return {entry['doc_id']: entry['tf'] for entry in index_local.get(term, [])}

def evaluate_and(query_terms: list[str]) -> list[tuple]:
    """
    Évalue une requête AND.
    Le score est la somme des TF des termes de la requête.
    """
    if not query_terms:
        return []
    
    # Récupère les dictionnaires de postings pour chaque terme
    postings_dicts = [postings_to_dict(term) for term in query_terms]
    
    # Si un terme n'est pas trouvé (dict vide), le AND échoue
    if any(len(p) == 0 for p in postings_dicts):
        return []
    
    # Commence avec l'ensemble des doc_id du premier terme
    common_docs = set(postings_dicts[0].keys())
    
    # Calcule l'intersection des ensembles de doc_id
    for p in postings_dicts[1:]:
        common_docs.intersection_update(p.keys())
    
    # Calcule les scores pour les documents communs
    scores = []
    for doc_id in common_docs:
        # Le score est la somme des TF pour ce document
        score = sum(p[doc_id] for p in postings_dicts)
        scores.append((doc_id, score))
        
    # Trie par score (décroissant), puis par doc_id (croissant)
    return sorted(scores, key=lambda x: (-x[1], x[0]))


def evaluate_or(query_terms: list[str]) -> list[tuple]:
    """
    Évalue une requête OR.
    Le score est la somme des TF des termes de la requête.
    """
    scores = defaultdict(int)
    
    # Pour chaque terme de la requête
    for term in query_terms:
        # Parcours sa liste de postings
        for doc_id, tf in postings_to_dict(term).items():
            # Ajoute le TF au score du document
            scores[doc_id] += tf
            
    # Trie par score (décroissant), puis par doc_id (croissant)
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))

# 4. Définir et exécuter les requêtes
sample_queries = [
    ["romeo", "juliet"],
    ["king", "queen"],
    ["thee", "thou"],
    ["love", "hate"],
    ["death", "sleep", "dream"]
]

print("Évaluation des requêtes...")
md_lines = ["# Résultats de la Recherche Booléenne", ""]

# Définir le nombre max de résultats à afficher par requête
TOP_K = 10

for terms in sample_queries:
    query_str = ' '.join(terms)
    and_hits = evaluate_and(terms)
    or_hits = evaluate_or(terms)
    
    md_lines.append(f"## Requête: `{query_str}`")
    md_lines.append("")
    
    # Section AND
    md_lines.append(f"### AND ({len(and_hits)} documents)")
    md_lines.append("DocID | Score (Sum TF)")
    md_lines.append("--- | ---")
    if and_hits:
        md_lines.extend(f"{doc} | {score}" for doc, score in and_hits[:TOP_K])
    else:
        md_lines.append("*(aucun résultat)*")
    md_lines.append("")

    # Section OR
    md_lines.append(f"### OR ({len(or_hits)} documents)")
    md_lines.append("DocID | Score (Sum TF)")
    md_lines.append("--- | ---")
    if or_hits:
        md_lines.extend(f"{doc} | {score}" for doc, score in or_hits[:TOP_K])
    else:
        md_lines.append("*(aucun résultat)*")
    md_lines.append("")
    md_lines.append("---")
    md_lines.append("")

# 5. Action: Écrire les résultats dans un fichier Markdown
queries_path = OUTPUTS_DIR / "queries_and_results.md"
newline = os.linesep
queries_path.write_text(newline.join(md_lines) + newline)

print(f"Résultats des requêtes sauvegardés dans {queries_path}")

Collecte de l'index en mémoire (driver)...


Index chargé avec 23512 termes.
Évaluation des requêtes...
Résultats des requêtes sauvegardés dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab2/assignments/outputs/queries_and_results.md


## 8. Part D — Performance study

In [10]:
# write some code here
# - vary spark.sql.shuffle.partitions and compare runtime and UI metrics
# - discuss pairs vs stripes trade-offs; include one explain('formatted') text in proof/


## 9. Spark UI evidence
Open http://localhost:4040 during runs. Capture Files Read, Input Size, Shuffle Read/Write and save screenshots under `proof/`.

## 10. Environment and reproducibility

In [11]:
# write some code here
# - print Java version, Spark conf, OS info
# - save ENV.md: versions + key configs
import json
import subprocess

def get_java_version():
    try:
        output = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT)
        return output.decode("utf-8").strip().splitlines()[0]
    except Exception as exc:
        return f"Unavailable ({exc})"

java_output = get_java_version()
print(f"Java: {java_output}")

print("Spark configuration (selected):")
conf_items = sorted(spark.sparkContext.getConf().getAll())
for key, value in conf_items:
    print(f" - {key} = {value}")

env_summary = {
    "python": sys.version,
    "spark": spark.version,
    "pyspark": pyspark.__version__,
    "java": java_output,
    "os": platform.platform(),
    "spark_conf": {k: v for k, v in conf_items if k.startswith("spark.")}}

env_lines = [
    "# Environment Summary",
    "",
    f"- Python: {sys.version.split()[0]}",
    f"- Spark: {spark.version}",
    f"- PySpark: {pyspark.__version__}",
    f"- Java: {java_output}",
    f"- OS: {platform.platform()}",
    "",
    "## Spark Configuration"]

env_lines.extend(f"- {k} = {v}" for k, v in env_summary["spark_conf"].items())

ENV_PATH = Path("ENV.md")
ENV_PATH.write_text("\n".join(env_lines) + "\n")

print(f"Environment details saved to {ENV_PATH.resolve()}")